<a href="https://colab.research.google.com/github/2000030914/2000030914/blob/main/acc%3D76.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# =========================
# IMPORTS
# =========================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [19]:
# =========================
# DATASET
# =========================
base_path = "/content/drive/MyDrive/AnnotatedUltrasoundLiver_Dataset"
classes = ['Benign', 'Malignant', 'Normal']

filepaths, labels = [], []

for label in classes:
    folder = os.path.join(base_path, label, "image")
    for file in os.listdir(folder):
        filepaths.append(os.path.join(folder, file))
        labels.append(label)

df = pd.DataFrame({'filename': filepaths, 'class': labels})

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['class'], random_state=42
)

In [20]:
# =========================
# GENERATORS (BEST SETTING)
# =========================
IMG_SIZE = 224
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse'
)

val_generator = val_datagen.flow_from_dataframe(
    val_df,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

Found 588 validated image filenames belonging to 3 classes.
Found 147 validated image filenames belonging to 3 classes.


In [21]:
# =========================
# MODEL: MOBILENET (FINAL)
# =========================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.BatchNormalization(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(3, activation='softmax')
])

In [22]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15
)

Epoch 1/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.4388 - loss: 1.6742 - val_accuracy: 0.6190 - val_loss: 0.9413
Epoch 2/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 37s 999ms/step - accuracy: 0.5391 - loss: 1.3315 - val_accuracy: 0.6735 - val_loss: 0.8025
Epoch 3/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 986ms/step - accuracy: 0.5782 - loss: 1.1003 - val_accuracy: 0.7007 - val_loss: 0.7452
Epoch 4/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6276 - loss: 0.9874 - val_accuracy: 0.7075 - val_loss: 0.7127
Epoch 5/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 37s 997ms/step - accuracy: 0.6344 - loss: 0.9604 - val_accuracy: 0.7007 - val_loss: 0.6865
Epoch 6/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6820 - loss: 0.8089 - val_accuracy: 0.7211 - val_loss: 0.6672
Epoch 7/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 36s 983ms/step - accuracy: 0.6633 - loss: 0.8638 - val_accuracy: 0.7211 - val_loss: 0.6700
Epoch 8/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.7177 - loss: 0.7342 - val_accuracy: 0.7279

In [25]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15
)

Epoch 1/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.7449 - loss: 0.6134 - val_accuracy: 0.7143 - val_loss: 0.8252
Epoch 2/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.7687 - loss: 0.5792 - val_accuracy: 0.7143 - val_loss: 0.8280
Epoch 3/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.7806 - loss: 0.5516 - val_accuracy: 0.7211 - val_loss: 0.8144
Epoch 4/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.7959 - loss: 0.5269 - val_accuracy: 0.7279 - val_loss: 0.7992
Epoch 5/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.7857 - loss: 0.5175 - val_accuracy: 0.7551 - val_loss: 0.7661
Epoch 6/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.8112 - loss: 0.4793 - val_accuracy: 0.7415 - val_loss: 0.7496
Epoch 7/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.8027 - loss: 0.4868 - val_accuracy: 0.7619 - val_loss: 0.7400
Epoch 8/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.8146 - loss: 0.4940 - val_accuracy: 0.7619 - val_loss:

In [26]:
# =========================
# EVALUATION
# =========================
val_loss, val_acc = model.evaluate(val_generator)
print("Final Accuracy:", val_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 627ms/step - accuracy: 0.7619 - loss: 0.7104
Final Accuracy: 0.761904776096344
